In [ ]:
# ¿Cómo podemos anticipar si una frase de una noticia financiera comunica una señal positiva, negativa o neutral?

# El archivo contiene frases financieras etiquetadas, por lo que permite vincular el texto con una tarea predictiva concreta.

import pandas as pd

dataframe = pd.read_csv(
    "../data/sentences.csv.gz",
    index_col=False,
    compression="gzip",
)

In [ ]:
# ¿Cuántas frases están disponibles para entrenar y evaluar el clasificador?

dataframe.shape

In [ ]:
# Se inspeccionan las variables que describen cada mensaje y su sentimiento observado.

dataframe.head()

In [ ]:
# ¿Las clases de sentimiento están balanceadas?

dataframe.target.value_counts()

In [ ]:
# Se revisan ejemplos de frases positivas para contextualizar la etiqueta.

for i in range(5):
    print(dataframe[dataframe.target == "positive"]["phrase"].iloc[i])

In [ ]:
# Se revisan ejemplos de frases negativas para contextualizar la etiqueta.

for i in range(5):
    print(dataframe[dataframe.target == "negative"]["phrase"].iloc[i])

In [ ]:
# Se revisan ejemplos de frases neutrales para contextualizar la etiqueta.

for i in range(5):
    print(dataframe[dataframe.target == "neutral"]["phrase"].iloc[i])

In [ ]:
# Se reservan mensajes de cada sentimiento para evaluar predicciones sobre datos no vistos.

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    dataframe.phrase,
    dataframe.target,
    test_size=0.3,
    random_state=0,
    stratify=dataframe.target,
)

In [ ]:
# Se construye una matriz documento-término que convierte cada frase en variables numéricas.

from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(
    lowercase=True,
    analyzer="word",
    token_pattern=r"\b[a-zA-Z]\w+\b",
    stop_words="english",
    max_df=0.99,
    min_df=2,
    binary=True,
)

X_train_vectorized = vectorizer.fit_transform(X_train)
vocabulary = vectorizer.get_feature_names_out()

print(f"Frases de entrenamiento: {X_train_vectorized.shape[0]}")
print(f"Términos del vocabulario: {X_train_vectorized.shape[1]}")
pd.DataFrame({"term": vocabulary[:20]})

In [ ]:
# Se ajusta un clasificador que asigne una probabilidad a cada sentimiento posible.

from sklearn.linear_model import LogisticRegression

clf = LogisticRegression(max_iter=1000, random_state=0)

clf.fit(
    X_train_vectorized,
    y_train,
)

In [ ]:
# Se compara el desempeño de entrenamiento y prueba para detectar una posible brecha de generalización.

from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score

# Precisión de entrenamiento:

accuracy_score(
    y_true=y_train,
    y_pred=clf.predict(X_train_vectorized),
)

In [ ]:
# La precisión por sí sola puede ocultar fallas en clases menos frecuentes.

X_test_vectorized = vectorizer.transform(X_test)
predictions = clf.predict(X_test_vectorized)

test_accuracy = accuracy_score(y_true=y_test, y_pred=predictions)
test_balanced_accuracy = balanced_accuracy_score(y_true=y_test, y_pred=predictions)
test_macro_f1 = f1_score(y_true=y_test, y_pred=predictions, average="macro")

test_accuracy, test_balanced_accuracy, test_macro_f1

In [ ]:
# La matriz muestra qué clases confunde el modelo y evita interpretar una única métrica como evidencia suficiente.

from sklearn.metrics import ConfusionMatrixDisplay

disp = ConfusionMatrixDisplay.from_predictions(
    y_test,
    predictions,
    cmap="Greens",
)
disp.figure_.suptitle("Matriz de confusión")
print(f"Matriz de confusión:\n{disp.confusion_matrix}")

In [ ]:
# Se guardan el modelo, el vectorizador y las métricas de prueba como evidencia verificable.

import json
import pickle

with open("../submission/clf.pkl", "wb") as file:
    pickle.dump(clf, file)

with open("../submission/vectorizer.pkl", "wb") as file:
    pickle.dump(vectorizer, file)

metrics = {
    "test_accuracy": test_accuracy,
    "test_balanced_accuracy": test_balanced_accuracy,
    "test_macro_f1": test_macro_f1,
    "test_size": len(y_test),
}

with open("../submission/metrics.json", "w", encoding="utf-8") as file:
    json.dump(metrics, file, indent=2)

In [ ]:
# Se comprueba que los objetos almacenados conservan las clases y permiten interpretar una predicción concreta.

with open("../submission/clf.pkl", "rb") as file:
    new_clf = pickle.load(file)

with open("../submission/vectorizer.pkl", "rb") as file:
    new_vectorizer = pickle.load(file)

new_predictions = new_clf.predict(new_vectorizer.transform(X_test))
new_predicted_proba = new_clf.predict_proba(new_vectorizer.transform(X_test))

results = pd.DataFrame(
    {
        "phrase": X_test.reset_index(drop=True),
        "observed_target": y_test.reset_index(drop=True),
        "predicted_target": new_predictions,
        "prediction_probability": new_predicted_proba.max(axis=1),
    }
)
results["prediction_probability"] = results["prediction_probability"].round(3)
results.head(10)